In [ ]:
/**
 * Design Database for Edtech Platform:
 * 1. Define Clear Requirements
 *    > We need to model a database for an educational platform involving:
 *      - Students registering for Courses
 *      - Instructors teaching Courses
 *      - Courses having additional detailed information
 *      - Assignments linked to courses
 *      - Departments categorizing courses and instructors
 * 
 * 2. Identify Entities
 *    - Student
 *    - Course
 *    - Instructor
 *    - Assignment
 *    - Department
 *    - Enrollment (Junction table for Many-to-Many Relationship)
 * 
 * 3. Define Relationships Clearly
 *    +------------+--------------+----------------+-------------------+
 *    | Entity A   | Relationship | Entity B       | Relationship Type |
 *    +------------+--------------+----------------+-------------------+
 *    | Instructor |   teaches    | Course         |    One-to-Many    |
 *    | Department |     has      | Courses        |    One-to-Many    |
 *    | Department |     has      | Instructor     |    One-to-Many    |
 *    |   Course   |     has      | Course details |    One-to-One     |
 *    |   Course   |     has      | Assignments    |    One-to-Many    |
 *    |  Student   |   enrolls    | Courses        |    Many-to-Many   |
 *    +------------+--------------+----------------+-------------------+
 * 
 * 4. Check Normalization Issues
 *       StudentsCourse(
 *          StudentID, StudentName, StudentEmail,
 *          CourseID, CourseName, 
 *          InstructorName, InstructorEmail, 
 *          DepartmentName,
 *          AssignmentTitles, 
 *          CourseSyllabus, 
 *       )
 * 
 *    a. Current 1NF Issues:
 *       > Rule: One Cell = One Value
 * 
 *       > Problem (Not 1NF): 
 *         - The column AssignmentTitles-likely contains multiple assignment titles in a 
 *           single cell.
 *         - Also, each row combines Student and Course info: a student enrolled in
 *           multiple courses will create repetition.
 *         +-----------+-------------+-----------------------+
 *         | StudentID | CourseName  | AssignmentTitles      |
 *         +-----------+-------------+-----------------------+
 *         | 1         | Java        | HW1, HW2, Project     |
 *         +-----------+-------------+-----------------------+
 * 
 * 
 *       > Fix (1NF): Split into rows
 *         - Students(StudentID, StudentName, StudentEmail)
 *         - Courses(CourseID, CourseName, InstructorName, InstructorEmail, DepartmentName, CourseSyllabus)
 *         - Enrollements(StudentID, CourseID)
 *         - Assignments(AssignmentID, CourseID, AssignmentTitle)
 * 
 *         a. Students Table:
 *            +-----------+-------------+----------------------+
 *            | StudentID | StudentName | StudentEmail         |
 *            +-----------+-------------+----------------------+
 *            | 1         | Mohammad    | mohammad@email.com   |
 *            +-----------+-------------+----------------------+
 *        
 *         b. Courses Table:
 *            +----------+-------------+----------------+--------------------+----------------+------------------+
 *            | CourseID | CourseName  | InstructorName | InstructorEmail    | DepartmentName | CourseSyllabus   |
 *            +----------+-------------+----------------+--------------------+----------------+------------------+
 *            | 101      | Java        | John           | john@email.com     | CS             | Basics of Java   |
 *            +----------+-------------+----------------+--------------------+----------------+------------------+
 *        
 *         c. Enrollments Table:
 *            +-----------+----------+
 *            | StudentID | CourseID |
 *            +-----------+----------+
 *            | 1         | 101      |
 *            +-----------+----------+
 *        
 *         d. Assignments Table:
 *            +--------------+----------+-----------------+
 *            | AssignmentID | CourseID | AssignmentTitle |
 *            +--------------+----------+-----------------+
 *            | 1            | 101      | HW1             |
 *            | 2            | 101      | HW2             |
 *            | 3            | 101      | Project         |
 *            +--------------+----------+-----------------+
 *        
 * 
 *    b. Current 2NF Issues:
 *       > Rule: If table has 2 keys (StudentID + CourseID),
 *               then any extra column must depend on BOTH keys.
 *       
 *       Step-1: Look at your table
 *       > Enrollments(StudentID, CourseID)
 *         +-----------+----------+
 *         | StudentID | CourseID |
 *         +-----------+----------+
 *         | 1         | 101      |
 *         +-----------+----------+
 *       > Primary Key = StudentID + CourseID
 * 
 *       Step-2: Check for extra columns
 *       > Do we have columns like: StudentName? CourseName?
 *         - No! Only StudentID and CourseID are present.
 *         - So, this table is perfect and already in 2NF
 *           - No extra data & nothing depends partially
 * 
 *       When 2NF Problem Happens?
 *       > Imagine this table:
 *         Enrollments(
 *           StudentID,
 *           CourseID,
 *           StudentName
 *         )
 *       > Problem:
 *         - StudentName depends only on StudentID
 *         - It does not need CourseID
 *         - So, it depends on HALF Key
 *       > Easy way to think:
 *         - Ask: "Does this col need BOTH StudentID and CourseID?"
 *         - If YES, OK
 *         - If NO, WRONG
 *       > Solution:
 *         - Move StudentName to Students table:
 *        
 *           Students(
 *             StudentID,
 *             StudentName
 *           )
 *          
 *           Enrollments(
 *             StudentID,
 *             CourseID
 *           )
 * 
 *         [2NF = No "half dependency"]
 * 
 * 
 *    c. Current 3NF Issue:
 *    > Rule:
 *      - Every column should depend ONLY on primary key
 *      - No column should depend on another column
 *   
 *    > STEP 1: CURRENT TABLES
 *      Students(StudentID, StudentName, StudentEmail)
 *     
 *      Courses(
 *        CourseID,
 *        CourseName,
 *        InstructorName,
 *        InstructorEmail,
 *        DepartmentName,
 *        CourseSyllabus
 *      )
 *     
 *      Enrollments(StudentID, CourseID)
 *     
 *      Assignments(AssignmentID, CourseID, AssignmentTitle)
 *   
 *   
 *    > STEP 2: FIND THE PROBLEM (FOCUS ON COURSES)
 *      - Ask: "Is this column directly about Course?"
 *        ✔  CourseName      → YES (depends on CourseID) ✅
 *        ✔  CourseSyllabus  → YES                       ✅
 *        ❌ InstructorName  → belongs to Instructor     ❌
 *        ❌ InstructorEmail → belongs to Instructor     ❌
 *        ❌ DepartmentName  → belongs to Department     ❌
 *   
 *   
 *    > PROBLEM (TRANSITIVE DEPENDENCY)
 *      - When one column depends on another column, and that column depends on the primary key.
 *        So dependency happens in chain: A -> B -> C, instead of: A -> C directly.
 *    
 *      - Example:
 *        CourseID → InstructorName → InstructorEmail
 *        CourseID → DepartmentName
 *        These columns depend on OTHER entities, not directly on CourseID ❌
 *   
 *   
 *    > SIMPLE THINKING TRICK
 *      - Ask: "Is this column describing the course OR something else?"
 *        - Course → keep in Courses table ✅
 *        - Instructor → move              ❌
 *        - Department → move              ❌
 *   
 *   
 *    > STEP 3: FIX (SPLIT TABLES)
 *      a. Instructors Table:
 *         +--------------+-----------------+----------------------+
 *         | InstructorID | InstructorName  | InstructorEmail      |
 *         +--------------+-----------------+----------------------+
 *   
 *      b. Departments Table:
 *         +--------------+----------------+
 *         | DepartmentID | DepartmentName |
 *         +--------------+----------------+
 *   
 *      c. Courses Table (clean now):
 *         +----------+-------------+--------------+--------------+------------------+
 *         | CourseID | CourseName  | InstructorID | DepartmentID | CourseSyllabus   |
 *         +----------+-------------+--------------+--------------+------------------+
 *   
 *    > 3NF = No indirect dependency (no "via another column")
 *   
 *    
 *    FINAL DESIGN (ALL TABLES): 
 *    - Students(
 *         StudentID [PK], 
 *         StudentName, 
 *         StudentEmail
 *      )
 *    - Instructors(
 *         InstructorID [PK], 
 *         InstructorName, 
 *         InstructorEmail,
 *         DepartmentID
 *      )
 *    - Departments(
 *         DepartmentID [PK], 
 *         DepartmentName
 *      )
 *    - Courses(
 *         CourseID [PK], 
 *         CourseName, 
 *         InstructorID [FK], 
 *         DepartmentID [FK], 
 *         CourseSyllabus
 *      )
 *    - Assignments(
 *         AssignmentID [PK], 
 *         CourseID     [FK], 
 *         AssignmentTitle
 *      )
 *    - Enrollments(
 *         StudentID [FK], 
 *         CourseID [FK]
 *      )
 * 
 * 5. Create Database Structure
 *    > Open: tablePlus
 *    > Click on: New Button > PostgreSQL:
 *      - BackendCohort
 *      - IP: default
 *      - Port: default
 *      - User: dockerPostgre-Username
 *      - Password: dockerPostgre-Password
 *    > Click on Test > Connect
 *    > Click on Create Database: New DB > edtech > Open
 * 
 *    a. Departments Table:
 *       CREATE TABLE departments (
 *          id INT AUTO_INCREMENT PRIMARY KEY,
 *          name VARCHAR(100) NOT NULL,
 *          
 *          created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
 *          updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
 *       )
 *    
 *    b. Instructors Table:
 *       CREATE TABLE instructors (
 *          id INT AUTO_INCREMENT PRIMARY KEY,
 *          name VARCHAR(100) NOT NULL,
 *          email VARCHAR(100) NOT NULL UNIQUE,
 *          department_id INT NOT NULL,
 *    
 *          created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
 *          updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
 *    
 *          CONSTRAINT fk_instructor_department
 *          FOREIGN KEY (department_id)
 *          REFERENCES departments(id)
 *          ON DELETE CASCADE
 *          ON UPDATE CASCADE
 *       )
 *    
 *    c. Students Table:
 *       CREATE TABLE students (
 *          id INT AUTO_INCREMENT PRIMARY KEY,
 *          name VARCHAR(100) NOT NULL,
 *          email VARCHAR(100) NOT NULL UNIQUE,
 *    
 *          created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
 *          updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
 *       )
 *    
 *    d. Courses Table:
 *       CREATE TABLE courses (
 *          id INT AUTO_INCREMENT PRIMARY KEY,
 *          name VARCHAR(100) NOT NULL,
 *          instructor_id INT NOT NULL,
 *          department_id INT NOT NULL,
 *          syllabus TEXT NOT NULL,
 *    
 *          created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
 *          updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
 *    
 *          CONSTRAINT fk_courses_instructor
 *          FOREIGN KEY (instructor_id)
 *          REFERENCES instructors(id)
 *          ON DELETE CASCADE
 *          ON UPDATE CASCADE,
 *    
 *          CONSTRAINT fk_courses_department
 *          FOREIGN KEY (department_id)
 *          REFERENCES departments(id)
 *          ON DELETE CASCADE
 *          ON UPDATE CASCADE
 *       )
 *    
 *    e. Enrollments Table:
 *       CREATE TABLE enrollments (
 *          student_id INT NOT NULL,
 *          course_id INT NOT NULL,
 *    
 *          PRIMARY KEY (student_id, course_id),
 *    
 *          created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
 *          updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
 *    
 *          CONSTRAINT fk_enrollment_student
 *          FOREIGN KEY (student_id)
 *          REFERENCES students(id)
 *          ON DELETE CASCADE
 *          ON UPDATE CASCADE,
 *    
 *          CONSTRAINT fk_enrollment_course
 *          FOREIGN KEY (course_id)
 *          REFERENCES courses(id)
 *          ON DELETE CASCADE
 *          ON UPDATE CASCADE
 *       );
 *    
 *    f. Assignment Table:
 *       CREATE TABLE assignments (
 *          id INT AUTO_INCREMENT PRIMARY KEY,
 *          course_id INT NOT NULL,
 *          title VARCHAR(255) NOT NULL,
 * 
 *          created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
 *          updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
 *    
 *          CONSTRAINT fk_assignment_course
 *          FOREIGN KEY (course_id)
 *          REFERENCES courses(id)
 *          ON DELETE CASCADE
 *          ON UPDATE CASCADE
 *       )
 * 
 * 6. Seeding the Database:
 * 
 * 
 * 7. Fetching Data:
 *    a. Get the list of all courses
 *       - SELECT * FROM courses;
 *       - SELECT id, name, syllabus FROM courses;
 *    b. Get all the departments
 *       - SELECT id, name FROM departments;
 *    c. Get one department (by ID) 
 *       - SELECT id, name FROM departments WHERE id = 3;
 * 
 * JOIN:
 * a. INNER JOIN:
 *    > Return only rows where there is a match in both tables.
 *    > Ex: Get all courses with instructors name (JOIN instructors table inside courses table):
 *      - SELECT c.id AS course_id, 
 *        c.name AS course_name, 
 *        c.syllabus AS course_syllabus,
 *        i.id AS instructor_id, 
 *        i.name AS instructor_name  
 *        FROM courses c
 *        INNER JOIN instructors i
 *        ON i.id = c.instructor_id;
 * 
 * b. LEFT JOIN:
 *    > Returns all rows from the left table (the one after FROM), and the matched rows
 *      from the right table. 
 *    > If there is no match, columns from the right table came back as NULL.
 *    > Ex:
 *      - SELECT d.name AS department_name, 
 *           i.name AS instructor_name, 
 *           i.email AS instructor_email,
 *        FROM departments d
 *        LEFT JOIN instructors i
 *        ON d.id = i.department_id;
 * 
 * d. RIGHT JOIN:
 *    > Returns all rows from the right table and the matched rows from the left table.
 *    > If no match, columns from the left table come back as NULL.
 *    > Ex: 
 *      - SELECT d.name AS department_name, 
 *           i.name AS instructor_name, 
 *           i.email AS instructor_email,
 *        FROM instructors i
 *        RIGHT JOIN departments d
 *        ON d.id = i.department_id;
 * 
 * d. FULL(OUTER) JOIN:
 *    > Returns all rows from both tables, matching them where possible, otherwise NULL
 *      for missing matches.
 * 
 * e. CROSS JOIN:
 *    > Produces a Cartesian product of the two tables i.e., every row of the table paired
 *      with every row of the right table.
 *    > Ex:
 * 
 * 
 * 8. SQL Assignments:
 *    a. List all departments (for an admin dashboard or dropdown).
 *    b. Get one department (by ID) with all its details.
 *    c. List all instructors in a department.
 *    d. List all courses offered in a department.
 *    e. List all instructors (could be used in an admin panel).
 *    f. Get a specific instructor by ID.
 *    g. Get all courses taught by a specific instructor.
 *    h. Find which department an instructor belongs to.
 *    i. List all students. 
 *    j. Get a student by ID.
 *    k. Find all courses a student is enrolled in.
 *    l. List all students not enrolled in any courses. 
 *    m. List all courses (show instructor name too). 
 *    n. Get a specific course (by ID) with department & instructor info. 
 *       - SELECT c.id AS course_id,
 *            c.name AS course_name,
 *            c.syllabus AS course_syllabus,
 *            i.name AS instructor_name,
 *            i.email AS instructor_email,
 *            d.name AS department_name
 *         FROM courses c
 *         INNER JOIN instructors i
 *         ON c.instructor_id = i.id 
 *         INNER JOIN departments d
 *         ON c.department_id = d.id
 *         WHERE c.id = 3;
 *    o. List all students in a particular course. 
 *       - SELECT s.id AS student_id
 *            s.name AS student_name,
 *         FROM enrollments e
 *         INNER JOIN students s
 *         ON e.student_id = s.id
 *         WHERE e.course_id = 3;
 *    p. Find how many students are enrolled in each course (aggregated). 
 *    q. List all enrollments (student/course pairs). 
 *    r. Enroll a student in a course (INSERT into pivot). 
 *    s. Unenroll a student from a course (DELETE from pivot). 
 *    t. List all assignments. 
 *    u. Get all assignments for a particular course. 
 *    v. Get assignment details (with course/instructor info). 
 *    w. Which instructor teaches the most courses? (count courses per instructor). 
 *       - SELECT i.id AS instructor_id,
 *            i.name AS instructor_name,
 *            COUNT(c.id) as total_courses
 *         FROM instructors i
 *         LEFT JOIN courses c
 *         ON i.id = c.instructor_id
 *         GROUP BY i.id, i.name
 *         ORDER BY total_courses DESC
 *         LIMIT 1;
 *    x. Which student is enrolled in the most courses? (count enrollments). 
 *    y. Courses with no students enrolled.
*/

In [ ]:
-- create departments table
CREATE TABLE departments (
	id INT AUTO_INCREMENT PRIMARY KEY,
	name VARCHAR(100) NOT NULL,

	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
	updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
);

-- instructor table
CREATE TABLE instructors (
	id INT AUTO_INCREMENT PRIMARY KEY,
	name VARCHAR(100) NOT NULL,
	email VARCHAR(100) NOT NULL UNIQUE,
	department_id INT NOT NULL,

	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
	updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP

	CONSTRAINT fk_instructor_department
	FOREIGN KEY (department_id)
	REFERENCES departments(id)
	ON DELETE CASCADE
	ON UPDATE CASCADE
);

-- students table
CREATE TABLE students (
	id INT AUTO_INCREMENT PRIMARY KEY,
	name VARCHAR(100) NOT NULL,
	email VARCHAR(100) NOT NULL UNIQUE,

	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
	updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
)

-- RENAME TABLE courses TO students;
-- ALTER TABLE courses RENAME TO students;

-- create courses table
CREATE TABLE courses (
	id INT AUTO_INCREMENT PRIMARY KEY,
	name VARCHAR(100) NOT NULL,
	instructor_id INT NOT NULL,
	department_id INT NOT NULL,

	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
	updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,

	CONSTRAINT fk_courses_instructor
	FOREIGN KEY (instructor_id)
	REFERENCES instructors(id)
	ON DELETE CASCADE
	ON UPDATE CASCADE,

	CONSTRAINT fk_courses_department
	FOREIGN KEY (department_id)
	REFERENCES deparments(id)
	ON DELETE CASCADE
	ON UPDATE CASCADE
);

-- create enrollments table
CREATE TABLE enrollments (
	student_id INT NOT NULL,
	course_id INT NOT NULL,

	PRIMARY KEY (student_id, course_id),

	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
	updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,

	CONSTRAINT fk_enrollement_student
	FOREIGN KEY (student_id)
	REFERENCES students(id)
	ON DELETE CASCADE
	ON UPDATE CASCADE,

	CONSTRAINT fk_enrollments_course
	FOREIGN KEY (course_id)
	REFERENCES courses(id)
	ON DELETE CASCADE
	ON UPDATE CASCADE
);


-- create assignments table
CREATE TABLE assignments (
	id INT AUTO_INCREMENT PRIMARY KEY,
	course_id INT NOT NULL,
	title VARCHAR(255) NOT NULL,

	CONSTRAINT fk_assignment_course
	FOREIGN KEY (course_id)
	REFERENCES courses(id)
	ON DELETE CASCADE
	ON UPDATE CASCADE
);

-- add columns to assignments table
ALTER TABLE assignments
ADD COLUMN 	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
ADD COLUMN  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP;

-- add syllabus column to courses table
ALTER TABLE courses
ADD COLUMN syllabus TEXT NOT NULL
AFTER name;


-- Seed the data

-- departments
INSERT INTO departments (name, created_at, updated_at)
VALUES
('Department 1', NOW(), NOW()),
('Department 2', NOW(), NOW()),
('Department 3', NOW(), NOW()),
('Department 4', NOW(), NOW()),
('Department 5', NOW(), NOW()),
('Department 6', NOW(), NOW()),
('Department 7', NOW(), NOW()),
('Department 8', NOW(), NOW()),
('Department 9', NOW(), NOW()),
('Department 10', NOW(), NOW()),
('Department 11', NOW(), NOW()),
('Department 12', NOW(), NOW()),
('Department 13', NOW(), NOW()),
('Department 14', NOW(), NOW()),
('Department 15', NOW(), NOW()),
('Department 16', NOW(), NOW()),
('Department 17', NOW(), NOW()),
('Department 18', NOW(), NOW()),
('Department 19', NOW(), NOW()),
('Department 20', NOW(), NOW());

-- instructors
INSERT INTO instructors (name, email, department_id, created_at, updated_at)
VALUES
('Instructor 1',  'instructor1@example.com',  1,  NOW(), NOW()),
('Instructor 2',  'instructor2@example.com',  2,  NOW(), NOW()),
('Instructor 3',  'instructor3@example.com',  3,  NOW(), NOW()),
('Instructor 4',  'instructor4@example.com',  4,  NOW(), NOW()),
('Instructor 5',  'instructor5@example.com',  5,  NOW(), NOW()),
('Instructor 6',  'instructor6@example.com',  6,  NOW(), NOW()),
('Instructor 7',  'instructor7@example.com',  7,  NOW(), NOW()),
('Instructor 8',  'instructor8@example.com',  8,  NOW(), NOW()),
('Instructor 9',  'instructor9@example.com',  9,  NOW(), NOW()),
('Instructor 10', 'instructor10@example.com', 10, NOW(), NOW()),
('Instructor 11', 'instructor11@example.com', 11, NOW(), NOW()),
('Instructor 12', 'instructor12@example.com', 12, NOW(), NOW()),
('Instructor 13', 'instructor13@example.com', 13, NOW(), NOW()),
('Instructor 14', 'instructor14@example.com', 14, NOW(), NOW()),
('Instructor 15', 'instructor15@example.com', 15, NOW(), NOW()),
('Instructor 16', 'instructor16@example.com', 16, NOW(), NOW()),
('Instructor 17', 'instructor17@example.com', 17, NOW(), NOW()),
('Instructor 18', 'instructor18@example.com', 18, NOW(), NOW()),
('Instructor 19', 'instructor19@example.com', 19, NOW(), NOW()),
('Instructor 20', 'instructor20@example.com', 20, NOW(), NOW());

-- students
INSERT INTO students (name, email, created_at, updated_at)
VALUES
('Student 1',  'student1@example.com',  NOW(), NOW()),
('Student 2',  'student2@example.com',  NOW(), NOW()),
('Student 3',  'student3@example.com',  NOW(), NOW()),
('Student 4',  'student4@example.com',  NOW(), NOW()),
('Student 5',  'student5@example.com',  NOW(), NOW()),
('Student 6',  'student6@example.com',  NOW(), NOW()),
('Student 7',  'student7@example.com',  NOW(), NOW()),
('Student 8',  'student8@example.com',  NOW(), NOW()),
('Student 9',  'student9@example.com',  NOW(), NOW()),
('Student 10', 'student10@example.com', NOW(), NOW()),
('Student 11', 'student11@example.com', NOW(), NOW()),
('Student 12', 'student12@example.com', NOW(), NOW()),
('Student 13', 'student13@example.com', NOW(), NOW()),
('Student 14', 'student14@example.com', NOW(), NOW()),
('Student 15', 'student15@example.com', NOW(), NOW()),
('Student 16', 'student16@example.com', NOW(), NOW()),
('Student 17', 'student17@example.com', NOW(), NOW()),
('Student 18', 'student18@example.com', NOW(), NOW()),
('Student 19', 'student19@example.com', NOW(), NOW()),
('Student 20', 'student20@example.com', NOW(), NOW());

-- courses
INSERT INTO courses (name, instructor_id, department_id, syllabus, created_at, updated_at)
VALUES
('Course 1',  1,  1,  'Syllabus for Course 1',  NOW(), NOW()),
('Course 2',  2,  2,  'Syllabus for Course 2',  NOW(), NOW()),
('Course 3',  3,  3,  'Syllabus for Course 3',  NOW(), NOW()),
('Course 4',  4,  4,  'Syllabus for Course 4',  NOW(), NOW()),
('Course 5',  5,  5,  'Syllabus for Course 5',  NOW(), NOW()),
('Course 6',  6,  6,  'Syllabus for Course 6',  NOW(), NOW()),
('Course 7',  7,  7,  'Syllabus for Course 7',  NOW(), NOW()),
('Course 8',  8,  8,  'Syllabus for Course 8',  NOW(), NOW()),
('Course 9',  9,  9,  'Syllabus for Course 9',  NOW(), NOW()),
('Course 10', 10, 10, 'Syllabus for Course 10', NOW(), NOW()),
('Course 11', 11, 11, 'Syllabus for Course 11', NOW(), NOW()),
('Course 12', 12, 12, 'Syllabus for Course 12', NOW(), NOW()),
('Course 13', 13, 13, 'Syllabus for Course 13', NOW(), NOW()),
('Course 14', 14, 14, 'Syllabus for Course 14', NOW(), NOW()),
('Course 15', 15, 15, 'Syllabus for Course 15', NOW(), NOW()),
('Course 16', 16, 16, 'Syllabus for Course 16', NOW(), NOW()),
('Course 17', 17, 17, 'Syllabus for Course 17', NOW(), NOW()),
('Course 18', 18, 18, 'Syllabus for Course 18', NOW(), NOW()),
('Course 19', 19, 19, 'Syllabus for Course 19', NOW(), NOW()),
('Course 20', 20, 20, 'Syllabus for Course 20', NOW(), NOW());

-- enrollements
INSERT INTO enrollments (student_id, course_id, created_at, updated_at)
VALUES
(1,  1,  NOW(), NOW()),
(2,  2,  NOW(), NOW()),
(3,  3,  NOW(), NOW()),
(4,  4,  NOW(), NOW()),
(5,  5,  NOW(), NOW()),
(6,  6,  NOW(), NOW()),
(7,  7,  NOW(), NOW()),
(8,  8,  NOW(), NOW()),
(9,  9,  NOW(), NOW()),
(10, 10, NOW(), NOW()),
(11, 11, NOW(), NOW()),
(12, 12, NOW(), NOW()),
(13, 13, NOW(), NOW()),
(14, 14, NOW(), NOW()),
(15, 15, NOW(), NOW()),
(16, 16, NOW(), NOW()),
(17, 17, NOW(), NOW()),
(18, 18, NOW(), NOW()),
(19, 19, NOW(), NOW()),
(20, 20, NOW(), NOW());

-- assignments
INSERT INTO assignments (course_id, title, created_at, updated_at)
VALUES
(1,  'Assignment for Course 1',   NOW(), NOW()),
(2,  'Assignment for Course 2',   NOW(), NOW()),
(3,  'Assignment for Course 3',   NOW(), NOW()),
(4,  'Assignment for Course 4',   NOW(), NOW()),
(5,  'Assignment for Course 5',   NOW(), NOW()),
(6,  'Assignment for Course 6',   NOW(), NOW()),
(7,  'Assignment for Course 7',   NOW(), NOW()),
(8,  'Assignment for Course 8',   NOW(), NOW()),
(9,  'Assignment for Course 9',   NOW(), NOW()),
(10, 'Assignment for Course 10',  NOW(), NOW()),
(11, 'Assignment for Course 11',  NOW(), NOW()),
(12, 'Assignment for Course 12',  NOW(), NOW()),
(13, 'Assignment for Course 13',  NOW(), NOW()),
(14, 'Assignment for Course 14',  NOW(), NOW()),
(15, 'Assignment for Course 15',  NOW(), NOW()),
(16, 'Assignment for Course 16',  NOW(), NOW()),
(17, 'Assignment for Course 17',  NOW(), NOW()),
(18, 'Assignment for Course 18',  NOW(), NOW()),
(19, 'Assignment for Course 19',  NOW(), NOW()),
(20, 'Assignment for Course 20',  NOW(), NOW());